# KBStats Matchday-Qualified Average Points

This notebook exports every KBStats player who meets the same matchday-dependent appearance rule used by the SofaScore player-average-ratings notebook. It applies no points, percentile, or average cutoff.

On matchday 1, a player qualifies with at least one played match. On matchdays 2–3, a player qualifies with at least two played matches, or by playing in the latest KBStats history slot. From matchday 4, a player qualifies with at least three played matches, or by playing in both of the two latest history slots. `history[0]` is the latest slot.

In [4]:
from __future__ import annotations

import json
import math
import re
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display


def locate_project_root() -> Path:
    starts = []
    notebook_path = globals().get('__vsc_ipynb_file__')
    if isinstance(notebook_path, str) and notebook_path.strip():
        starts.append(Path(notebook_path).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / 'project_paths.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate project_paths.py. Start Jupyter from the project root.')


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    DERIVED_KBSTATS_MATCHDAY_QUALIFIED_AVERAGE_PLAYERS_DIR,
    KBSTATS_PLAYERS_DIR,
    ensure_directory,
)

MATCHDAY_ONE_MINIMUM_PLAYED_MATCHES = 1
EARLY_SEASON_MAX_MATCHDAY = 3
EARLY_SEASON_MINIMUM_PLAYED_MATCHES = 2
MINIMUM_PLAYED_MATCHES = 3
MINIMUM_BUNDESLIGA_MATCHDAY = 1
MAXIMUM_BUNDESLIGA_MATCHDAY = 34
KBSTATS_FILENAME_RE = re.compile(
    r'^kbstats_players_(?P<date>\d{8})_(?P<time>\d{6})_(?P<offset>[+-]\d{4})\.json$'
)


def prompt_bundesliga_matchday() -> int:
    raw_matchday = input(f'Current Bundesliga matchday ({MINIMUM_BUNDESLIGA_MATCHDAY}-{MAXIMUM_BUNDESLIGA_MATCHDAY}): ').strip()
    try:
        matchday = int(raw_matchday)
    except ValueError as exc:
        raise ValueError('Bundesliga matchday must be a whole number.') from exc
    if not MINIMUM_BUNDESLIGA_MATCHDAY <= matchday <= MAXIMUM_BUNDESLIGA_MATCHDAY:
        raise ValueError(f'Bundesliga matchday must be between {MINIMUM_BUNDESLIGA_MATCHDAY} and {MAXIMUM_BUNDESLIGA_MATCHDAY}.')
    return matchday


BUNDESLIGA_MATCHDAY = prompt_bundesliga_matchday()


Current Bundesliga matchday (1-34):  1


In [5]:
def parse_snapshot_timestamp(path: Path) -> datetime:
    match = KBSTATS_FILENAME_RE.fullmatch(path.name)
    if match is None:
        raise ValueError(f'Unsupported KBStats filename: {path.name}')
    parsed = datetime.strptime(
        f"{match.group('date')}_{match.group('time')}_{match.group('offset')}",
        '%Y%m%d_%H%M%S_%z',
    )
    return parsed.astimezone(timezone.utc)


def select_latest_snapshot(directory: Path) -> Path:
    if not directory.is_dir():
        raise FileNotFoundError(f'KBStats player output directory not found: {directory}')
    candidates = []
    for path in sorted(directory.glob('kbstats_players_*.json')):
        try:
            candidates.append((parse_snapshot_timestamp(path), path))
        except ValueError as exc:
            warnings.warn(f'Ignoring {path.name}: {exc}', stacklevel=2)
    if not candidates:
        raise FileNotFoundError(f'No valid kbstats_players_*.json files in {directory}')
    latest = max(timestamp for timestamp, _ in candidates)
    latest_paths = [path for timestamp, path in candidates if timestamp == latest]
    if len(latest_paths) != 1:
        raise RuntimeError('Multiple KBStats files encode the same latest instant.')
    return latest_paths[0]


def is_finite_number(value: Any) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(float(value))


def played_in_history_slot(player: dict[str, Any], slot_index: int) -> bool:
    history = player.get('history')
    return bool(
        isinstance(history, list)
        and len(history) > slot_index
        and isinstance(history[slot_index], dict)
        and history[slot_index].get('hasPlayed') is True
    )


def qualification_rule_description(matchday: int) -> str:
    if matchday == MINIMUM_BUNDESLIGA_MATCHDAY:
        return 'at least 1 played match'
    if matchday <= EARLY_SEASON_MAX_MATCHDAY:
        return 'at least 2 played matches or played in the latest history slot'
    return 'at least 3 played matches or played in both latest history slots'


def player_qualifies(player: dict[str, Any], matchday: int) -> bool:
    games_played = player.get('gamesPlayed')
    if not is_finite_number(games_played):
        return False
    if matchday == MINIMUM_BUNDESLIGA_MATCHDAY:
        return games_played >= MATCHDAY_ONE_MINIMUM_PLAYED_MATCHES
    if matchday <= EARLY_SEASON_MAX_MATCHDAY:
        return games_played >= EARLY_SEASON_MINIMUM_PLAYED_MATCHES or played_in_history_slot(player, 0)
    return games_played >= MINIMUM_PLAYED_MATCHES or (played_in_history_slot(player, 0) and played_in_history_slot(player, 1))


selected_input_path = select_latest_snapshot(KBSTATS_PLAYERS_DIR)
try:
    raw_players = json.loads(selected_input_path.read_text(encoding='utf-8'))
except (OSError, UnicodeDecodeError, json.JSONDecodeError) as exc:
    raise ValueError(f'Could not load KBStats input {selected_input_path}: {exc}') from exc
if not isinstance(raw_players, list):
    raise TypeError(f'Expected a JSON list of players, got {type(raw_players).__name__}.')

valid_players, invalid_records = [], []
for source_index, player in enumerate(raw_players):
    if not isinstance(player, dict):
        invalid_records.append({'source_index': source_index, 'reason': 'record is not an object'})
    elif not is_finite_number(player.get('gamesPlayed')) or not is_finite_number(player.get('averagePoints')):
        invalid_records.append({'source_index': source_index, 'player_id': player.get('id'), 'reason': 'gamesPlayed or averagePoints is not finite numeric data'})
    else:
        valid_players.append(player)

qualified_players = sorted(
    (player for player in valid_players if player_qualifies(player, BUNDESLIGA_MATCHDAY)),
    key=lambda player: (-float(player['averagePoints']), -float(player['gamesPlayed']), str(player.get('name') or '').casefold()),
)

display_df = pd.DataFrame([
    {'player_id': player.get('id'), 'name': player.get('name'), 'team_id': player.get('teamId'), 'position': player.get('position'), 'average_points': player.get('averagePoints'), 'games_played': player.get('gamesPlayed'), 'total_points': player.get('totalPoints'), 'market_value': player.get('marketValue')}
    for player in qualified_players
])
print(f'Selected input: {selected_input_path.name}')
print(f'Matchday {BUNDESLIGA_MATCHDAY} rule: {qualification_rule_description(BUNDESLIGA_MATCHDAY)}')
print(f'Source records: {len(raw_players):,}; valid records: {len(valid_players):,}; excluded invalid records: {len(invalid_records):,}; qualified players: {len(qualified_players):,}')
display(display_df)


Selected input: kbstats_players_20260818_204143_+0200.json
Matchday 1 rule: at least 2 played matches or played in the latest history slot
Source records: 468; valid records: 310; excluded invalid records: 158; qualified players: 304


,player_id,name,team_id,position,average_points,games_played,total_points,market_value
0,8329,Michael Olise,2,3,225.0,32.0,7185.0,64621231.0
1,12368,Sander Tangvik,6,1,220.0,1.0,220.0,515772.0
2,7226,Harry Kane,2,4,216.0,31.0,6703.0,68674466.0
3,1685,Joshua Kimmich,2,3,186.0,29.0,5391.0,59703646.0
4,11675,Luis Díaz,2,4,178.0,32.0,5698.0,53334606.0
...,...,...,...,...,...,...,...,...
299,12333,Youssoupha Niang,28,4,6.0,10.0,55.0,5090221.0
300,7325,Elias Baum,4,2,6.0,5.0,32.0,10030248.0
301,7350,Otto Stange,6,4,5.0,10.0,51.0,3023502.0
302,4318,Tim Oermann,7,2,2.0,2.0,3.0,5762724.0


In [6]:
generated_datetime = datetime.now().astimezone()
output_timestamp = generated_datetime.strftime('%Y%m%d_%H%M%S_%z')
output_directory = ensure_directory(DERIVED_KBSTATS_MATCHDAY_QUALIFIED_AVERAGE_PLAYERS_DIR)
json_output_path = output_directory / f'kbstats_matchday_qualified_average_players_{output_timestamp}.json'
csv_output_path = output_directory / f'kbstats_matchday_qualified_average_players_{output_timestamp}.csv'
output_document = {
    'generated_at': generated_datetime.isoformat(timespec='seconds'),
    'source_file': selected_input_path.name,
    'eligibility': {'bundesliga_matchday': BUNDESLIGA_MATCHDAY, 'early_season_max_matchday': EARLY_SEASON_MAX_MATCHDAY, 'rule': qualification_rule_description(BUNDESLIGA_MATCHDAY)},
    'source_player_count': len(raw_players),
    'valid_player_count': len(valid_players),
    'excluded_invalid_player_count': len(invalid_records),
    'qualified_player_count': len(qualified_players),
    'players': qualified_players,
}
json_output_path.write_text(json.dumps(output_document, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
csv_df = pd.json_normalize(qualified_players, sep='.')
for column in csv_df.columns:
    csv_df[column] = csv_df[column].map(lambda value: json.dumps(value, ensure_ascii=False) if isinstance(value, (list, dict)) else value)
csv_df.to_csv(csv_output_path, index=False, encoding='utf-8-sig')
print(f'JSON output: {json_output_path}')
print(f'CSV output:  {csv_output_path}')


JSON output: C:\kickbase project\outputs\derived\kbstats_matchday_qualified_average_players\kbstats_matchday_qualified_average_players_20260819_091135_+0200.json
CSV output:  C:\kickbase project\outputs\derived\kbstats_matchday_qualified_average_players\kbstats_matchday_qualified_average_players_20260819_091135_+0200.csv


In [ ]:
from project_paths import prune_timestamped_outputs

removed_outputs = prune_timestamped_outputs()
print(f"Pruned {len(removed_outputs)} expired timestamped output(s).")
